***

Preparing Workspace

***

In [ ]:
# General
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import math
import seaborn as sns

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.offline import plot
import plotly.subplots as sp
from plotly.subplots import make_subplots
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

if user == 'jfontes':
    # Git
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')

path_config = os.path.join(path_git, 'Data', 'FFIEC', 'config')
path_plots = r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring"


In [ ]:
## User defined functions
exec(open(os.path.join(path_config0,       'Functions.py')).read())
exec(open(os.path.join(path_config , 'FFIEC_functions.py')).read())

***

Burden_1

***

In [ ]:
indicator_name = 'Burden_1'

file_name = f"{indicator_name} MPO HMDA.xlsx"
df_mpo = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'MPO')

display(df_mpo.head())

In [ ]:
# Set Indicator
indicator_name = 'Burden_1'
plot_name = 'mortgage_lending'
export = False



## Importing ---


file_name = f"{indicator_name} MPO HMDA.xlsx"
df_mpo = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'MPO')


## Organizing ---


df_plot = df_mpo.copy()

df_plot = df_plot[['year', 'MPO', 'purpose', 'demographic', 'origination_rate']]
df_plot = df_plot.pivot_table(index = ['year', 'MPO', 'purpose']
                                            , columns = 'demographic'
                                            , values = 'origination_rate').reset_index()

df_plot['Female to Male'                              ] = df_plot['Female'                   ] - df_plot['Male'                  ]
df_plot['Hispanic or Latino to Not Hispanic or Latino'] = df_plot['Hispanic or Latino'       ] - df_plot['Not Hispanic or Latino']
df_plot['Non-White to White'                          ] = df_plot['Non-White'                ] - df_plot['White'                 ]
df_plot['Asian to White'                              ] = df_plot['Asian'                    ] - df_plot['White'                 ]
df_plot['Black or African American to White'          ] = df_plot['Black or African American'] - df_plot['White'                 ]

df_plot = df_plot[['year', 'MPO', 'purpose',
                 'Female to Male', 'Hispanic or Latino to Not Hispanic or Latino', 
                 'Non-White to White', 'Asian to White', 'Black or African American to White']]

df_plot = pd.melt(df_gap, id_vars = ['year', 'MPO', 'purpose'], var_name = 'demographic', value_name = 'origination_rate')
df_plot['origination_rate'] = df_plot['origination_rate']*100


df_plot['origination_rate'] = round(df_plot['origination_rate'], 1)
df_plot = df_plot[df_plot['purpose'    ] == 'All'  ]
df_plot = df_plot[df_plot['county_name'] == 'SACOG']



## Plotting ---


color_map = {
    'Female to Male': '#9DC209'
    , 'Hispanic or Latino to Not Hispanic or Latino': '#1E90FF'
    , 'Non-White to White': "#FBB117"
    , 'Asian to White': "#DC381F"
    , 'Black or African American to White': '#1F45FC'
}

fig = px.line(df_plot, x='year', y='origination_rate'
              , color='demographic'
              , color_discrete_map=color_map
              , markers=True)

title = '<b>Mortgage Loan Origination Gap</b>  <br><sup>6-County Sacramento Region</sup> '
fig.update_xaxes(dtick=1)
fig.update_traces(hovertemplate='%{y}')

plot_agol(export=export)